# Animagine XL 4.0 Studio (Optimized for Google Colab T4)
ตรวจสอบให้แน่ใจว่าเลือก Runtime เป็น **T4 GPU** แล้ว (*เมนูด้านบน: Runtime -> Change runtime type -> T4 GPU*)

In [ ]:
#@title 1. ติดตั้งไลบรารีที่จำเป็น (กดรันครั้งเดียว)
!pip -q install diffusers transformers accelerate scipy safetensors xformers gradio ipywidgets==7.7.1

In [ ]:
#@title 2. เลือกและโหลดโมเดล (กดรันแล้วคลิกปุ่ม Apply)
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler, DDIMScheduler, EulerDiscreteScheduler, UniPCMultistepScheduler
from diffusers.models import AutoencoderKL
import torch, gc
import ipywidgets as widgets
from google.colab import output

output.enable_custom_widget_manager()
pipe = None

select_model = widgets.Dropdown(
    options=[
        ("Animagine XL 4.0", "cagliostrolab/animagine-xl-4.0"),
        ("Stable Diffusion XL Base 1.0", "stabilityai/stable-diffusion-xl-base-1.0"),
        ("SSD-1B (Fast SDXL)", "segmind/SSD-1B"),
    ],
    value="cagliostrolab/animagine-xl-4.0",
    description="Select Model:"
)

select_sampler = widgets.Dropdown(
    options=[
        "EulerAncestralDiscreteScheduler",
        "EulerDiscreteScheduler",
        "UniPCMultistepScheduler",
        "DDIMScheduler"
    ],
    description="Select Scheduler:"
)
select_sampler.style.description_width = "auto"

out = widgets.Output()
apply_btn = widgets.Button(description="Apply", button_style="info")

def get_scheduler(name):
    schedulers = {
        "EulerAncestralDiscreteScheduler": EulerAncestralDiscreteScheduler,
        "DDIMScheduler": DDIMScheduler,
        "EulerDiscreteScheduler": EulerDiscreteScheduler,
        "UniPCMultistepScheduler": UniPCMultistepScheduler,
    }
    return schedulers[name].from_pretrained(select_model.value, subfolder="scheduler")

def pipeline(p):
    global pipe
    out.clear_output()
    apply_btn.disabled = True
    with out:
        print("กำลังโหลดโมเดล กรุณารอสักครู่...")
        try:
            if pipe is not None:
                del pipe
                gc.collect()
                torch.cuda.empty_cache()

            vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
            pipe = StableDiffusionXLPipeline.from_pretrained(
                select_model.value,
                scheduler=get_scheduler(select_sampler.value),
                torch_dtype=torch.float16,
                use_safetensors=True,
                low_cpu_mem_usage=True,
                vae=vae
            ).to("cuda")

            pipe.enable_xformers_memory_efficient_attention()
            pipe.vae.enable_tiling()
            pipe.vae.enable_slicing()

            print("โหลดโมเดลเสร็จสมบูรณ์! พร้อมใช้งานในขั้นตอนที่ 3")
        except Exception as e:
            print(f"Error: {e}")
    apply_btn.disabled = False

apply_btn.on_click(pipeline)

widgets.VBox([
    widgets.HTML(value="<h3>Configure Pipeline</h3>"),
    select_model, select_sampler, apply_btn, out
])

In [ ]:
#@title 3. เปิดหน้าต่างวาดภาพ (Gradio UI)
import gradio as gr
import torch
import random

def generate_images(prompt, negative_prompt, width, height, steps, cfg_scale, num_images, seed):
    if pipe is None:
        raise gr.Error("กรุณากดปุ่ม Apply ในขั้นตอนที่ 2 เพื่อโหลดโมเดลก่อนครับ!")

    if seed == -1:
        seed = random.randint(0, 2147483647)

    generator = torch.Generator("cuda").manual_seed(int(seed))

    with torch.inference_mode():
        images = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=int(width),
            height=int(height),
            num_inference_steps=int(steps),
            guidance_scale=float(cfg_scale),
            num_images_per_prompt=int(num_images),
            generator=generator
        ).images

    torch.cuda.empty_cache()
    return images, f"Seed: {seed}"

size_list = [768, 832, 1024, 1216, 1344]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Animagine XL 4.0 Studio")

    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label="Prompt",
                lines=4,
                value="1other, tiger, solo, safe, majestic tiger, detailed fur, jungle, forest background, cinematic lighting, vibrant colors, anime style, masterpiece, high score, great score, absurdres"
            )
            neg_prompt = gr.Textbox(
                label="Negative Prompt",
                lines=2,
                value="lowres, bad anatomy, bad hands, text, error, missing finger, extra digits, fewer digits, cropped, worst quality, low quality, low score, bad score, average score, signature, watermark, username, blurry"
            )

            with gr.Row():
                width = gr.Dropdown(label="Width", choices=size_list, value=832)
                height = gr.Dropdown(label="Height", choices=size_list, value=1216)

            with gr.Row():
                steps = gr.Slider(label="Steps", minimum=15, maximum=50, step=1, value=25)
                cfg = gr.Slider(label="CFG Scale", minimum=1.0, maximum=12.0, step=0.5, value=5.0)

            with gr.Row():
                num_images = gr.Slider(label="Batch Size", minimum=1, maximum=4, step=1, value=1)
                seed = gr.Number(label="Seed (-1 = Random)", value=-1, precision=0)

            btn = gr.Button("Generate", variant="primary")

        with gr.Column(scale=1):
            gallery = gr.Gallery(label="Gallery", columns=2, height="auto")
            seed_info = gr.Textbox(label="Info", interactive=False)

    btn.click(
        fn=generate_images,
        inputs=[prompt, neg_prompt, width, height, steps, cfg, num_images, seed],
        outputs=[gallery, seed_info]
    )

demo.launch(share=True, inline=True)